# SpatioAI — Phase 3: Temporal Extreme Weather Event Tracking
This notebook demonstrates:
1. Ingesting candidate extreme weather events across timestamps.
2. Multi-feature data association using geodesic distances, speed gating, bounding box IoU, area ratio, and intensity similarity.
3. Lifecycle management (track creation, update, gap bridging with `max_missed_steps`, and termination).
4. Visualizing temporal trajectories and tracking metrics on the Indian subcontinent map.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Setup project path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.events.association import EventAssociationConfig
from src.events.tracking import BaselineEventTracker
from scripts.validate_tracking_pipeline import create_synthetic_event_sequences

In [ ]:
# 1. Generate multi-event sequence
df_events = create_synthetic_event_sequences()
print(f"Loaded {len(df_events)} event detections across {df_events['timestamp'].nunique()} timestamps:")
display(df_events.head(6))

In [ ]:
# 2. Run Baseline Event Tracker
config = EventAssociationConfig(
    max_centroid_distance_km=500.0,
    max_speed_kmh=120.0,
    max_area_change_ratio=3.0,
    max_missed_steps=1,
    algorithm="hungarian",
)

tracker = BaselineEventTracker(config=config)
df_track_events, df_track_summary = tracker.track(df_events)

print(f"Total Tracks Formed: {len(df_track_summary)}")
display(df_track_summary)

In [ ]:
# 3. Plot Spatio-Temporal Trajectories and Bounding Boxes
fig, ax = plt.subplots(figsize=(12, 8))

colors = ["#d95f02", "#1b9e77", "#7570b3", "#e7298a"]

for idx, trk_id in enumerate(df_track_events["track_id"].unique()):
    trk_data = df_track_events[df_track_events["track_id"] == trk_id].sort_values("step_in_track")
    c = colors[idx % len(colors)]
    
    # Plot trajectory path
    ax.plot(trk_data["centroid_lon"], trk_data["centroid_lat"], color=c, marker="o", linewidth=2.5, markersize=8, label=f"{trk_id} Trajectory")
    
    # Plot bounding boxes and labels per step
    for _, ev in trk_data.iterrows():
        rect = plt.Rectangle(
            (ev["min_lon"], ev["min_lat"]),
            ev["max_lon"] - ev["min_lon"],
            ev["max_lat"] - ev["min_lat"],
            fill=False,
            edgecolor=c,
            linestyle="--",
            alpha=0.6,
        )
        ax.add_patch(rect)
        ax.text(ev["centroid_lon"] + 0.15, ev["centroid_lat"], f"{ev['event_id']}\n({ev['max_intensity']}mm)", color=c, fontsize=8)

ax.set_title("SpatioAI — Temporal Extreme Weather Event Tracks & Trajectories", fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude (°E)", fontsize=11)
ax.set_ylabel("Latitude (°N)", fontsize=11)
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="upper right", fontsize=11)

plt.show()